In [1]:
import numpy as np

def robust_fit_stats(X_raw_3d, eps=1e-9):
    """
    입력:
        X_raw_3d: (N,C,L) 또는 (N,L,C)
    출력:
        med: (C,)
        iqr: (C,)
    """

    if X_raw_3d.ndim != 3:
        raise ValueError(f"Expected 3D input, got {X_raw_3d.shape}")

    # (N,C,L)
    if X_raw_3d.shape[1] == 13:
        X2 = np.transpose(X_raw_3d, (0, 2, 1)).reshape(-1, X_raw_3d.shape[1])

    # (N,L,C)
    elif X_raw_3d.shape[2] == 13:
        X2 = X_raw_3d.reshape(-1, X_raw_3d.shape[2])

    else:
        raise ValueError(f"Cannot find channel dim=13 in shape {X_raw_3d.shape}")

    med = np.median(X2, axis=0)
    q1 = np.percentile(X2, 25, axis=0)
    q3 = np.percentile(X2, 75, axis=0)

    iqr = q3 - q1
    iqr[iqr < eps] = eps

    return med, iqr

def robust_transform_chunk(X_raw_3d, med, iqr, clip_k=5.0, map01=True, chunk=50000):
    N = X_raw_3d.shape[0]
    C = X_raw_3d.shape[1]
    L = X_raw_3d.shape[2]

    med = med.reshape(1, C, 1)
    iqr = iqr.reshape(1, C, 1)

    X_out = np.empty((N, C, L), dtype=np.float32)

    for i in range(0, N, chunk):
        j = min(i + chunk, N)

        X = X_raw_3d[i:j].astype(np.float32)

        X = (X - med) / iqr
        X = np.clip(X, -clip_k, clip_k)

        if map01:
            X = (X + clip_k) / (2.0 * clip_k)

        X_out[i:j] = X

    return X_out

def ensure_ncl(X, C=12):
    """Return X in (N, C, L)."""
    if X.ndim != 3:
        raise ValueError(f"X must be 3D, got {X.ndim}D")
    if X.shape[1] == C:
        return X  # already (N,C,L)
    if X.shape[2] == C:
        return np.transpose(X, (0, 2, 1))  # (N,L,C) -> (N,C,L)
    raise ValueError(f"Cannot find channel dim={C} in shape {X.shape}")



In [2]:
import numpy as np

dataset_npz_path = "D:/IDS_masters/dataset/carchallenge_test_0305_537.npz"
test_path = "D:/IDS_masters/dataset/carchallenge_test_training_0305_537.npz"

# ----------------------------
# Load
# ----------------------------
train_data = np.load(dataset_npz_path)
test_data  = np.load(test_path)


for ch in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,10,11,12]:  # ch1=0, ch2=1, ch10=9 (0-indexed)
    tr = train_data["X"][:, ch, :].flatten()
    te = test_data["X"][:, ch, :].flatten()
    print(f"ch{ch+1} | Train: mean={tr.mean():.4f} std={tr.std():.4f}")
    print(f"ch{ch+1} | Test:  mean={te.mean():.4f} std={te.std():.4f}")

X_train, y_train = train_data["X"], train_data["y"]
X_test,  y_test  = test_data["X"],  test_data["y"]

# ----------------------------
# Force same layout: (N,C,L)
# ----------------------------
X_train = ensure_ncl(X_train, C=13)
X_test  = ensure_ncl(X_test,  C=13)

# (선택) 길이도 동일한지 확인
L_train = X_train.shape[2]
L_test  = X_test.shape[2]
if L_train != L_test:
    raise ValueError(f"Sequence length mismatch: train L={L_train}, test L={L_test}")

# y shape sanity (Task가 sequence label이면 (N,L)이어야 함)
# 여기서는 너의 파이프라인이 y를 (N,L)로 쓰는 것으로 보였으니 체크만 걸어둠
if y_train.ndim == 2 and y_train.shape[1] != L_train:
    raise ValueError(f"y_train length mismatch: y_train.shape={y_train.shape}, L_train={L_train}")
if y_test.ndim == 2 and y_test.shape[1] != L_test:
    raise ValueError(f"y_test length mismatch: y_test.shape={y_test.shape}, L_test={L_test}")

# ----------------------------
# Robust scaling (fit on TRAIN only)
# ----------------------------
med, iqr = robust_fit_stats(X_train)


# 1) robust
X_train_r = robust_transform_chunk(X_train, med, iqr)
X_test_r  = robust_transform_chunk(X_test,  med, iqr)

save_train = "D:/IDS_masters/dataset/carchallenge_robust_0305_X.npz"
save_test        = "D:/IDS_masters/dataset/carchallenge_training_robust_0305_537.npz"


# 저장
np.savez(save_train, X=X_train_r.astype(np.float32), y=y_train)
np.savez(save_test,        X=X_test_r.astype(np.float32),  y=y_test)

print("Saved scaled datasets to same paths.")

ch1 | Train: mean=0.0511 std=0.2202
ch1 | Test:  mean=0.0466 std=0.2107
ch2 | Train: mean=0.9391 std=0.1479
ch2 | Test:  mean=0.9381 std=0.1487
ch3 | Train: mean=0.1054 std=0.0810
ch3 | Test:  mean=0.1068 std=0.0801
ch4 | Train: mean=0.7252 std=0.5392
ch4 | Test:  mean=0.7348 std=0.5350
ch5 | Train: mean=0.0270 std=0.0433
ch5 | Test:  mean=0.0263 std=0.0416
ch6 | Train: mean=0.7265 std=0.0533
ch6 | Test:  mean=0.7288 std=0.0518
ch7 | Train: mean=0.0116 std=0.0453
ch7 | Test:  mean=0.0105 std=0.0434
ch8 | Train: mean=1.7469 std=2.1700
ch8 | Test:  mean=1.7088 std=2.0827
ch9 | Train: mean=-0.5485 std=0.7692
ch9 | Test:  mean=-0.4897 std=0.7465
ch10 | Train: mean=0.0495 std=0.0664
ch10 | Test:  mean=0.0460 std=0.0641
ch11 | Train: mean=0.0867 std=0.1654
ch11 | Test:  mean=0.0832 std=0.1595
ch12 | Train: mean=0.2104 std=0.3240
ch12 | Test:  mean=0.1637 std=0.3111
ch13 | Train: mean=2.2130 std=1.1166
ch13 | Test:  mean=2.2333 std=1.1130
Saved scaled datasets to same paths.
